# Train, evaluate, and track a fraud model

**Goal.** Build a leakage-safe model comparison from a bounded payment run.

**Audience.** ML engineers validating a fraud feature pipeline.

**Prerequisites.** `poetry install -E ml`; no external service is required.

**Produces.** Inspection tables, temporal splits, metrics, a local artifact, and a manifest.

**Source size.** The default cells generate approximately 1,000 logical payments; increase duration and population together for a 10,000-payment run.

**Offline path.** All marked offline cells run without Docker or network services. Service cells are optional and explicitly marked in notebook metadata.

**Cleanup.** Outputs are written under a temporary directory; remove any local run directory if you changed the output location.


**Set up a deterministic source run**


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import polars as pl

from fraudtwin.config import load_config
from fraudtwin.generation import generate

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs" / "minimal.yaml").exists()),
    Path.cwd(),
)
base = load_config(root / "configs" / "minimal.yaml")
# Scale the population so the bounded example produces about 1,000 payments.
population = base.population.model_copy(
    update={
        "customers": 200,
        "accounts": 300,
        "cards": 240,
        "devices": 240,
        "pix_keys": 160,
        "merchants": 60,
    }
)
simulation = base.simulation.model_copy(update={"duration_days": 10})
fraud = base.fraud.model_copy(update={"enabled": True, "target_rate": 0.05})
config = base.model_copy(
    update={"population": population, "simulation": simulation, "fraud": fraud}
)
data = generate(config, write=False)
run_id = data.run_id
payments = pl.DataFrame([item.model_dump(mode="json") for item in data.behavior.payments])
print({"run_id": run_id, "payments": len(payments), "events": len(data.behavior.payment_events)})

**Inspect schema, grain, and counts**


In [ ]:
dataset = data.require_dataset()
fraud_ids = {item.payment_id for item in data.behavior.fraud_records if item.fraud_truth}
rows = [
    dict(row, label=("FRAUD" if row["payment_id"] in fraud_ids else "LEGITIMATE"))
    for row in dataset.rows
]
frame = dataset.frame.with_columns(pl.Series("label", [row["label"] for row in rows]))
print({"rows": dataset.count, "splits": frame.group_by("split").len().to_dicts()})
display(
    frame.select(["split", "label"])
    .group_by("split")
    .agg(pl.len().alias("rows"), pl.col("label").eq("FRAUD").sum().alias("fraud"))
)
assert "prediction_time" in frame.columns

**Run the core operation**


In [ ]:
# Check temporal boundaries and obvious leakage indicators before fitting.
times = frame.select(
    pl.col("prediction_time").min().alias("first"), pl.col("prediction_time").max().alias("last")
)
print(times)
assert "prediction_time" in frame.columns and "label" in frame.columns
print("leakage check: only point-in-time features are used; labels are evaluation targets")

**Measure and interpret the result**


In [ ]:
from fraudtwin.ml import heuristic_predictions, load_baseline_config, train_baselines

policy = load_baseline_config(root / "configs" / "ml-baselines.yaml")
heuristic = heuristic_predictions(rows)
print({"heuristic_predictions": len(heuristic), "policy": policy.model_dump(mode="json")})
try:
    trained = train_baselines(rows, policy, source_run_dir=None)
    print("trained models:", list(trained.model_artifacts or {}))
except (RuntimeError, ValueError) as exc:
    fallback_policy = policy.model_copy(update={"models": ("deterministic_heuristic",)})
    trained = train_baselines(rows, fallback_policy)
    print("optional model training unavailable; deterministic fallback trained:", exc)

**Exercise a parameter or failure mode**


In [ ]:
from fraudtwin.ml import evaluate_predictions

heuristic_result = evaluate_predictions(rows, heuristic, policy, model_id="heuristic")
metrics = pl.DataFrame(list(heuristic_result.metrics))
display(metrics)
print(
    "available metric names:",
    sorted(metrics["metric"].unique().to_list()) if "metric" in metrics else metrics.columns,
)

**Write a compact artifact and fingerprint**


In [ ]:
with TemporaryDirectory(prefix="fraudtwin-tutorial-09-") as tmp:
    out = Path(tmp)
    manifest = out / "evaluation-manifest.json"
    manifest.write_text(
        json.dumps(heuristic_result.manifest, indent=2, default=str), encoding="utf-8"
    )
    print({"artifact": str(manifest), "bytes": manifest.stat().st_size})

**Verify invariants and clean up**


In [ ]:
summary = {
    "run_id": run_id,
    "payments": len(data.behavior.payments),
    "payment_events": len(data.behavior.payment_events),
    "fraud_records": len(data.behavior.fraud_records),
}
assert summary["payments"] == len(payments)
assert summary["payments"] > 0
print(json.dumps(summary, indent=2, default=str))

## Record the generated shape and tutorial contract.


In [ ]:
active = next(
    (globals().get(name) for name in ("data", "baseline") if globals().get(name) is not None), None
)
assert active is not None
summary = {
    "tutorial_id": 9,
    "payments": len(active.behavior.payments),
    "events": len(active.behavior.payment_events),
}
print(summary)
assert summary["payments"] >= 0

## Inspect stable payment identities.


In [ ]:
ids = [item.payment_id for item in active.behavior.payments]
assert len(ids) == len(set(ids))
print({"unique_payment_ids": len(ids)})

## Record the generated shape and tutorial contract.


In [ ]:
active = next(
    (globals().get(name) for name in ("data", "baseline") if globals().get(name) is not None), None
)
assert active is not None
summary = {
    "tutorial_id": 9,
    "payments": len(active.behavior.payments),
    "events": len(active.behavior.payment_events),
}
print(summary)
assert summary["payments"] >= 0